# NB13 — Sensitivity analyses
Each variant re-fits the Cox model in every cohort and re-runs the fixed-effect meta-analysis. The frozen files are
**never** changed; variants are computed in memory. For every variant the M criterion of H1 is re-checked
(HR > 1 in all cohorts, fixed-effect p < 0.05, I2 < 50%).

| Variant | What changes |
|---|---|
| primary_m3 | reference (same as NB11) |
| m2_no_neuron | neuron-identity covariate removed |
| purity_proxy_1_only / purity_proxy_2_only | one purity proxy instead of two |
| neuron_markers_strict | neuron score without TUBB3, MAP2, STMN2 (also made by glioma cells) |
| M_detect_0.05 / M_detect_0.20 / M_patients_0.3 / M_patients_0.7 | M and R re-derived with other single-cell thresholds |
| LOO_without_<cohort> | one cohort left out |
| CGGA_treatment_MGMT | CGGA only, adds MGMT methylation, radiotherapy, TMZ |
| TCGA_incl_IDH_unknown | TCGA adults not sequenced for IDH added back (most TCGA GBM are IDH-wildtype) |
| S_alt_GOCC_synapse | S replaced by GO CC "synapse" (S only; M/R not defined for this set) |

Also: a rank-based (Stouffer) check of the NB05 DE meta-analysis, as NB05 promised.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from gbm_extra import load_expr
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
stop_if_smoke("NB13 needs the full results.")
verify_frozen()
VAL = P("validation"); SEN = P("sensitivity"); MAT = P("meta", "matrices"); SC = CFG["sc"]; H = CFG["survival"]["h1"]
S = pd.read_csv(P("final", "S_candidate_genes.csv")).gene.astype(str).tolist()
M0 = pd.read_csv(P("final", "M_genes.csv")).gene.astype(str).tolist()
R0 = pd.read_csv(P("final", "R_genes.csv")).gene.astype(str).tolist()
S_alt = pd.read_csv(P("final", "S_alt_GOCC_SYNAPSE.csv")).gene.astype(str).tolist()
mk = CFG["markers"]; neuron = [g for g in mk["neuron_identity"] if g not in set(S)]
neuron_strict = [g for g in neuron if g not in {"TUBB3", "MAP2", "STMN2"}]
cohorts = [c for c in ["TCGA", "CGGA693", "CGGA325"] if (VAL / f"{c}_scores.csv").exists()]
FILES = {"TCGA": MAT / "TCGA_logcpm.csv.gz", "CGGA693": MAT / "CGGA693_log2fpkm.csv.gz", "CGGA325": MAT / "CGGA325_log2fpkm.csv.gz"}
r_cox = write_r("cox_run.R", R_COX)
ADJ = {"m1": "age,sex_male", "m2": "age,sex_male,purity_proxy_1,purity_proxy_2",
       "m3": "age,sex_male,purity_proxy_1,purity_proxy_2,neuron_identity"}
base = {c: pd.read_csv(VAL / f"{c}_scores.csv") for c in cohorts}
expr = {c: load_expr(FILES[c]) for c in cohorts}
print({c: len(d) for c, d in base.items()}, "| neuron markers:", neuron, "| strict:", neuron_strict)

In [ ]:
def zscores(c, ids):
    e = expr[c][[s for s in ids if s in expr[c].columns]]
    return zscore_df(e[e.std(axis=1) > 0], axis=1)


def prep(d):
    d = d.copy()
    d["time"] = pd.to_numeric(d.os_time_days, errors="coerce"); d["event"] = pd.to_numeric(d.os_event, errors="coerce")
    d["sex_male"] = d.sex.astype(str).str.strip().str.lower().map(lambda s: 1.0 if s.startswith("m") else (0.0 if s.startswith("f") else np.nan))
    d["age"] = pd.to_numeric(d.age, errors="coerce")
    d = d[(d.time > 0) & d.event.isin([0, 1])].copy()
    for c in [x for x in ["S", "M", "R", "S_alt", "purity_proxy_1", "purity_proxy_2", "neuron_identity"] if x in d]:
        d[c] = (d[c] - d[c].mean()) / d[c].std()
    return d


def fit(variant, tables, adj, preds=("S", "M", "R")):
    rows = []
    pf = SEN / "preds.txt"; pf.write_text("\n".join(preds))
    for c, d in tables.items():
        dp = SEN / f"_tmp_{c}.csv"; prep(d).to_csv(dp, index=False)
        out = SEN / f"_tmp_{c}_out.csv"
        run_r(r_cox, [dp, pf, adj, out])
        t = pd.read_csv(out)
        if len(t):
            t["cohort"] = c; rows.append(t)
    cx = pd.concat(rows) if rows else pd.DataFrame()
    res = []
    for p in preds:
        sub = cx[cx.predictor == p] if len(cx) else cx
        if len(sub) < 2:
            continue
        r = dl_meta(sub.logHR.values[None, :], (sub.SE.values ** 2)[None, :])
        p_fixed = 2 * stats.norm.sf(abs(r["mu_fe"][0] / r["se_fe"][0]))
        all_gt1 = bool((sub.logHR > 0).all())
        res.append({"variant": variant, "predictor": p, "adjusted_for": adj, "cohorts": ",".join(sub.cohort),
                    "n_total": int(sub.n.sum()), "events_total": int(sub.events.sum()),
                    "HR_fixed": float(np.exp(r["mu_fe"][0])), "HR_lo": float(np.exp(r["mu_fe"][0] - 1.96 * r["se_fe"][0])),
                    "HR_hi": float(np.exp(r["mu_fe"][0] + 1.96 * r["se_fe"][0])), "p_fixed": p_fixed,
                    "HR_random": float(np.exp(r["mu"][0])), "p_random": float(r["p"][0]), "I2": float(r["I2"][0]),
                    "all_cohorts_HR_gt_1": all_gt1,
                    "M_criterion_met": bool(all_gt1 and p_fixed < H["meta_p"] and r["I2"][0] < H["i2_max"]),
                    "M_criterion_met_random": bool(all_gt1 and r["p"][0] < H["meta_p"])})
        for _, x in sub.iterrows():
            res[-1][f"HR_{x.cohort}"] = float(np.exp(x.logHR))
    return res


results = []
results += fit("primary_m3", base, ADJ["m3"])
results += fit("primary_m1", base, ADJ["m1"])
results += fit("m2_no_neuron", base, ADJ["m2"])
results += fit("purity_proxy_1_only", base, "age,sex_male,purity_proxy_1,neuron_identity")
results += fit("purity_proxy_2_only", base, "age,sex_male,purity_proxy_2,neuron_identity")
print(pd.DataFrame(results)[["variant", "predictor", "HR_fixed", "p_fixed", "I2", "M_criterion_met"]].round(3).to_string(index=False))

In [ ]:
# strict neuron markers
t_strict = {}
for c, d in base.items():
    z = zscores(c, d.sample_id)
    d2 = d.copy(); d2["neuron_identity"] = score_geneset(z, neuron_strict).reindex(d2.sample_id).values
    t_strict[c] = d2
results += fit("neuron_markers_strict", t_strict, ADJ["m3"])

# M / R re-derived with other single-cell thresholds
det = pd.read_csv(P("sc", "S_gene_detection_by_patient.csv"), index_col=0)
for lab, frac, pfrac in [("M_detect_0.05", 0.05, SC["min_patient_fraction"]), ("M_detect_0.20", 0.20, SC["min_patient_fraction"]),
                         ("M_patients_0.3", SC["detect_fraction_in_patient"], 0.3), ("M_patients_0.7", SC["detect_fraction_in_patient"], 0.7)]:
    ok = (det >= frac).mean(axis=1) >= pfrac
    Mv = sorted(ok[ok].index); Rv = sorted(set(S) - set(Mv))
    if len(Mv) < 10 or len(Rv) < 10:
        print(lab, "skipped: M or R < 10 genes"); continue
    tabs = {}
    for c, d in base.items():
        z = zscores(c, d.sample_id)
        d2 = d.copy()
        d2["M"] = score_geneset(z, Mv).reindex(d2.sample_id).values; d2["R"] = score_geneset(z, Rv).reindex(d2.sample_id).values
        tabs[c] = d2
    out = fit(lab, tabs, ADJ["m3"])
    for o in out:
        o["n_M_genes"] = len(Mv); o["overlap_with_frozen_M"] = len(set(Mv) & set(M0))
    results += out
    print(lab, "| M genes:", len(Mv), "| overlap with frozen M:", len(set(Mv) & set(M0)))

# leave one cohort out (needs >= 3 cohorts so 2 remain)
if len(cohorts) >= 3:
    for c in cohorts:
        results += fit(f"LOO_without_{c}", {k: v for k, v in base.items() if k != c}, ADJ["m3"])

# CGGA only with treatment and MGMT
cg = {}
for c in [x for x in cohorts if x.startswith("CGGA")]:
    d = base[c].copy()
    m = d.mgmt.astype(str).str.strip().str.lower()
    d["mgmt_meth"] = np.where(m == "methylated", 1.0, np.where(m.str.contains("un"), 0.0, np.nan))
    cg[c] = d
if len(cg) >= 2:
    results += fit("CGGA_treatment_MGMT", cg, ADJ["m3"] + ",mgmt_meth,radio,chemo")
    results += fit("CGGA_only_primary_m3", cg, ADJ["m3"])

# TCGA with IDH-unknown adults added back
if "TCGA" in cohorts:
    sh = pd.read_csv(P("meta", "TCGA_samples.csv"))
    extra = sh[(sh.idh_status.astype(str) == "unknown") & (pd.to_numeric(sh.age, errors="coerce") >= 18) & sh.os_time_days.notna()]
    if len(extra):
        ids = list(base["TCGA"].sample_id) + [s for s in extra.sample_id if s in expr["TCGA"].columns]
        z = zscores("TCGA", ids)
        sh_i = sh.drop_duplicates("sample_id").set_index("sample_id").loc[[s for s in ids if s in z.columns]]
        d = pd.DataFrame({"sample_id": sh_i.index})
        for lab, genes in [("S", S), ("M", M0), ("R", R0), ("neuron_identity", neuron), ("immune_stromal", mk["immune_stromal"]),
                           ("oligodendrocyte", mk["oligodendrocyte"]), ("astrocyte_normal", mk["astrocyte_normal"])]:
            d[lab] = score_geneset(z, genes).reindex(d.sample_id).values
        d["purity_proxy_1"] = -d.immune_stromal; d["purity_proxy_2"] = -(d.oligodendrocyte + d.astrocyte_normal) / 2
        d = d.merge(sh_i[["age", "sex", "os_time_days", "os_event"]].reset_index(), on="sample_id")
        results += fit("TCGA_incl_IDH_unknown", {**base, "TCGA": d}, ADJ["m3"])
        print("TCGA_incl_IDH_unknown: added", len(d) - len(base["TCGA"]), "patients")
    else:
        print("TCGA_incl_IDH_unknown: no IDH-unknown adults with OS; skipped.")

# alternative synaptic set (S only)
alt = {}
for c, d in base.items():
    z = zscores(c, d.sample_id)
    d2 = d.copy(); d2["S_alt"] = score_geneset(z, S_alt).reindex(d2.sample_id).values
    alt[c] = d2
results += fit("S_alt_GOCC_synapse_m1", alt, ADJ["m1"], preds=("S_alt",))
results += fit("S_alt_GOCC_synapse_m3", alt, ADJ["m3"], preds=("S_alt",))

for f in SEN.glob("_tmp_*"):
    f.unlink()
sens = pd.DataFrame(results)
sens.to_csv(SEN / "sensitivity_results.csv", index=False)
show = ["variant", "predictor", "cohorts", "HR_fixed", "HR_lo", "HR_hi", "p_fixed", "I2", "M_criterion_met"]
print(sens[show].round(3).to_string(index=False))

In [ ]:
# ---- rank-based (Stouffer) check of the DE meta-analysis ---------------------------------------------------
ma = pd.read_csv(P("meta_analysis", "meta_analysis_full.csv"))
studies = [c[len("logFC_"):] for c in ma.columns if c.startswith("logFC_")]
summ = pd.read_csv(P("de", "DE_summary_all_cohorts.csv")).set_index("cohort")
zs, ws = [], []
for k in studies:
    de = pd.read_csv(P("de", f"{k}_DE_full.csv")).drop_duplicates("gene").set_index("gene").reindex(ma.gene)
    zs.append(stats.norm.isf(de["P.Value"].clip(1e-300, 1) / 2) * np.sign(de.t))
    ws.append(np.sqrt(summ.loc[k, "n_gbm"] + summ.loc[k, "n_control"]) if k in summ.index and "n_gbm" in summ.columns else 1.0)
Z = np.column_stack(zs); W = np.array(ws)
ma["stouffer_z"] = (Z * W).sum(1) / np.sqrt((W ** 2).sum())
ma["stouffer_p"] = 2 * stats.norm.sf(np.abs(ma.stouffer_z))
agree = (np.sign(ma.stouffer_z) == np.sign(ma.meta_log2FC))
rk = pd.DataFrame([{"genes": "all", "n": len(ma), "sign_agreement": float(agree.mean()),
                    "spearman_rank_DL_vs_Stouffer": float(stats.spearmanr(ma.meta_log2FC / ma.meta_SE, ma.stouffer_z)[0])},
                   {"genes": "S", "n": int(ma.is_S.sum()), "sign_agreement": float(agree[ma.is_S].mean()),
                    "spearman_rank_DL_vs_Stouffer": float(stats.spearmanr((ma.meta_log2FC / ma.meta_SE)[ma.is_S], ma.stouffer_z[ma.is_S])[0])}])
rk.to_csv(SEN / "DE_meta_DL_vs_Stouffer.csv", index=False)
print(rk.round(3).to_string(index=False))

In [ ]:
def hr_axis(ax):
    """Readable log-scale HR axis: plain numbers, no overlapping minor labels."""
    from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
    lo, hi = ax.get_xlim()
    ticks = [t for t in [0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8] if lo <= t <= hi] or [1]
    ax.xaxis.set_major_locator(FixedLocator(ticks)); ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))


mm = sens[sens.predictor == "M"].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(6.5, 0.38 * len(mm) + 1.2))
for i, r in mm.iloc[::-1].reset_index(drop=True).iterrows():
    col = "#1f3a93" if r.M_criterion_met else "#9aa5b1"
    ax.errorbar(r.HR_fixed, i, xerr=[[r.HR_fixed - r.HR_lo], [r.HR_hi - r.HR_fixed]], fmt="o", color=col, ms=4, capsize=2)
ax.set_yticks(range(len(mm))); ax.set_yticklabels(mm.variant.iloc[::-1], fontsize=8)
ax.axvline(1, c="k", lw=.6, ls=":"); ax.set_xscale("log"); hr_axis(ax); ax.set_xlabel("M score HR per SD (fixed-effect meta, 95% CI)")
ax.set_title("Sensitivity of the M association (blue = H1 M criterion met)", fontsize=10)
savefig(fig, "Sensitivity_M_forest", folder="sensitivity"); plt.close(fig)
summary = {"n_variants_M": int(len(mm)), "n_M_criterion_met": int(mm.M_criterion_met.sum()),
           "variants_failing": mm.loc[~mm.M_criterion_met, "variant"].tolist()}
manifest_update("sensitivity", "summary", summary)
print(summary)
print("\nNB13 complete. Next: NB14 (integration).")